# v24 — Grid Search Menyeluruh: Cari Sinyal Terbaik untuk Zona Transisi/Ranging

**Latar belakang:** v23 menguji BOS M5+H1 dgn 1 kombinasi parameter tebakan (SL=2x/TP=3x ATR,
zona ADX 18-25, tanpa syarat momentum) di seluruh histori 2019-2026 -- hasilnya GAGAL jelas:
win rate 22.4%, PF 0.52, rugi $391 (n=277). Di TRAIN (2019-2023, sample lebih besar n=175):
win rate cuma 10.9%, PF 0.17. **User benar mengkritik ini**: 1 kombinasi tebakan bukan bukti
BOS+H1 gagal secara umum -- perlu dicari scr sistematis kombinasi yang benar2 cocok.

**Cakupan grid search (4 dimensi, semua diminta user)**:
1. **SL/TP multiplier ATR** -- SL 1.0-3.0x, TP 2.0-6.0x (beda dari trend kuat yg optimal 2x/4x,
   kondisi transisi mungkin butuh rasio berbeda)
2. **Rentang ADX transisi** -- geser batas bawah/atas (bukan cuma 18-25 tetap), coba beberapa
   kombinasi (adx_ranging_max, adx_trend_min)
3. **Syarat momentum chain minimum** -- BOS+H1 dikombinasikan dgn bull_chain/bear_chain >= N,
   supaya breakout yg ditangkap juga py momentum minimal (bukan structure break doang)
4. **max_hold** -- durasi tahan posisi transisi mungkin beda optimal dari trend kuat (12 candle)

**Perbaikan metodologi dari v23**: `adx_trend_min` di v23 accidentally di-set 25.0 (bukan 18.0
spt v13 asli) utk mode TREND -- di sini DIPERBAIKI kembali ke 18.0 (v13 asli) supaya baseline
pembanding benar2 valid, TIDAK bercampur dgn eksperimen zona transisi.

**Kriteria kejujuran (SAMA persis spt v19-v23)**: kandidat dianggap layak kalau PF>1.5 DI KEDUA
TRAIN & TEST, DAN sample TRAIN>=30, TEST>=15. Kalau tidak ada yang lolos, itu kesimpulan JUJUR
"belum ketemu" -- bukan dipaksakan pilih yang "kurang jelek".

**TIDAK ADA perubahan ke `usecase.py`** -- murni riset. Reuse cache v23
(`df_2019_2026_full_mtf.parquet`, sudah ada v12_score + OB + BOS M5/H1 + H1 EMA).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v24"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v23 (skor v12 + OB + BOS M5/H1 + H1 EMA, 2019-2026) + tambah bull_chain/bear_chain

In [2]:
v23_cache = PROCESSED_DIR / "v23" / "df_2019_2026_full_mtf.parquet"
assert v23_cache.exists(), "Cache v23 belum ada -- jalankan v23 dulu sblm v24"
df = pd.read_parquet(v23_cache)
print(f"Load dari cache v23: {len(df)} candle, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom: {list(df.columns)}")

# bull_chain/bear_chain sudah ada di cache v23 (dicek dari markdown v23 Section 1 keep_cols)
assert "bull_chain" in df.columns and "bear_chain" in df.columns, "Kolom momentum chain hilang dari cache"

Load dari cache v23: 518403 candle, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom: ['datetime', 'open', 'high', 'low', 'close', 'adx', 'atr', 'v12_score', 'bull_chain', 'bear_chain', 'bos_bull', 'bos_bear', 'ob_bull', 'ob_bear', 'h1_ob_bull', 'h1_ob_bear', 'h1_ema_50', 'h1_ema_200', 'h1_bos_bull', 'h1_bos_bear']


## 2. Backtest engine v24 -- semua parameter zona transisi jadi argumen grid search

In [3]:
def check_h1_alignment_v24(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v24(
    df_signals: pd.DataFrame,
    adx_trend_min: float = 18.0,      # v13 ASLI pakai 18.0, BUKAN 25.0 (bug v23 diperbaiki di sini)
    adx_transition_min: float = 12.0, # batas bawah zona transisi BOS_H1
    adx_transition_max: float = 18.0, # batas atas zona transisi (harus <= adx_trend_min)
    min_signal_score: float = 9.0,
    sl_mult_trend: float = 2.0,
    tp_mult_trend: float = 4.0,
    sl_mult_transition: float = 2.0,
    tp_mult_transition: float = 3.0,
    min_chain_transition: float = 0.0,  # syarat momentum chain minimum utk mode transisi, 0=tidak ada syarat
    max_hold_trend: int = 12,
    max_hold_transition: int = 12,
    enable_transition: bool = True,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    bos_bull_arr = df_signals["bos_bull"].to_numpy()
    bos_bear_arr = df_signals["bos_bear"].to_numpy()
    h1_bos_bull_arr = df_signals["h1_bos_bull"].to_numpy()
    h1_bos_bear_arr = df_signals["h1_bos_bear"].to_numpy()
    bull_chain_arr = df_signals["bull_chain"].to_numpy()
    bear_chain_arr = df_signals["bear_chain"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        direction = None
        mode = None
        sl_mult = tp_mult = max_hold = None

        if adx >= adx_trend_min:
            if score >= min_signal_score:
                direction = "BUY"
            elif score <= -min_signal_score:
                direction = "SELL"
            if direction is not None:
                if require_ob_filter:
                    opposing_ob = (
                        (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                        (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
                    )
                    if opposing_ob:
                        direction = None
                if direction is not None and require_h1_alignment:
                    if not check_h1_alignment_v24(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                        direction = None
            mode = "TREND"
            sl_mult, tp_mult, max_hold = sl_mult_trend, tp_mult_trend, max_hold_trend

        elif adx_transition_min <= adx < adx_transition_max and enable_transition:
            cand_dir = None
            if bos_bull_arr[i] == 1 and h1_bos_bull_arr[i] == 1:
                cand_dir = "BUY"
            elif bos_bear_arr[i] == 1 and h1_bos_bear_arr[i] == 1:
                cand_dir = "SELL"
            if cand_dir is not None and min_chain_transition > 0:
                dom_chain = bull_chain_arr[i] if cand_dir == "BUY" else bear_chain_arr[i]
                if dom_chain < min_chain_transition:
                    cand_dir = None
            if cand_dir is not None:
                direction, mode = cand_dir, "TRANSITION"
            sl_mult, tp_mult, max_hold = sl_mult_transition, tp_mult_transition, max_hold_transition

        if direction is None:
            i += 1
            continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v24 siap.")

Backtest engine v24 siap.


## 3. TRAIN/TEST split & verifikasi baseline v13 (adx_trend_min diperbaiki ke 18.0)

In [4]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_v13_train = run_backtest_v24(df_train, enable_transition=False)
trades_v13_test = run_backtest_v24(df_test, enable_transition=False)
print("\n=== v13 murni (adx_trend_min=18.0, BENAR spt live) ===")
print("TRAIN:", evaluate(trades_v13_train, INITIAL_EQUITY))
print("TEST :", evaluate(trades_v13_test, INITIAL_EQUITY))

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle



=== v13 murni (adx_trend_min=18.0, BENAR spt live) ===
TRAIN: {'total_trades': 2282, 'win_rate_pct': 15.69, 'profit_factor': np.float64(0.38), 'net_pnl': np.float64(-2348.28), 'max_drawdown_pct': np.float64(-2348.28)}
TEST : {'total_trades': 1105, 'win_rate_pct': 45.7, 'profit_factor': np.float64(1.58), 'net_pnl': np.float64(1588.28), 'max_drawdown_pct': np.float64(-229.55)}


## 4. Grid search menyeluruh -- cari kombinasi TRANSITION terbaik (isolasi mode TRANSITION saja)

4 dimensi sesuai permintaan user: SL/TP mult, rentang ADX transisi, syarat momentum chain
minimum, max_hold. Dievaluasi TERISOLASI (`enable_transition` aktif tapi lihat metrik mode
TRANSITION saja, bukan gabungan dgn TREND) supaya kualitas sinyalnya sendiri jelas terlihat.

In [5]:
import itertools
import time as _time

GRID = {
    "adx_transition_min": [10.0, 12.0, 15.0],
    "adx_transition_max": [16.0, 18.0, 20.0, 22.0],
    "sl_mult_transition": [1.0, 1.5, 2.0, 3.0],
    "tp_mult_transition": [2.0, 3.0, 4.0, 6.0],
    "min_chain_transition": [0.0, 2.0, 3.0, 4.0],
    "max_hold_transition": [6, 12, 20],
}
FIXED = dict(adx_trend_min=18.0, enable_transition=True)

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    if params["adx_transition_min"] >= params["adx_transition_max"]:
        continue
    trades = run_backtest_v24(df_train, **params, **FIXED)
    transition_only = trades[trades["mode"] == "TRANSITION"]
    metrics = evaluate(transition_only, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 200 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s ({len(grid_df)} kombinasi valid dari {len(combos)})")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 15 kandidat TRANSITION (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(15).to_string(index=False))
print(f"\nTotal kandidat dgn sample cukup: {len(grid_valid)} dari {len(grid_df)}")
print(f"Kandidat dgn PF > 1.5: {(grid_valid['profit_factor'] > 1.5).sum()}")

Total kombinasi grid: 2304


  [200/2304] 172s


  [400/2304] 348s


  [600/2304] 523s


  [800/2304] 698s


...

  [1600/2304] 1403s


  [1800/2304] 1580s


  [2000/2304] 1757s


  [2200/2304] 1933s



Grid search selesai dalam 2025s (2304 kombinasi valid dari 2304)

=== Top 15 kandidat TRANSITION (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  adx_transition_min  adx_transition_max  sl_mult_transition  tp_mult_transition  min_chain_transition  max_hold_transition
           30         26.67           0.32   -54.17          -2202.91                10.0                18.0                 3.0                 3.0                   3.0                   20
           30         26.67           0.32   -54.17          -2202.91                10.0                18.0                 3.0                 3.0                   4.0                   20
           30         26.67           0.32   -54.17          -2202.91                10.0                18.0                 3.0                 3.0                   2.0                   20
           30         26.67           0.32   -54.17          -2202.91                10.0                

## 5. Validasi TEST out-of-sample (semua kandidat TRAIN yang PF>1.5, bukan cuma yang #1)

Kriteria kejujuran: validasi SEMUA kandidat yang lolos threshold TRAIN, bukan cuma pilih 1
kandidat top -- supaya kelihatan apakah keberhasilan itu konsisten di banyak kombinasi
(robust) atau cuma 1 kombinasi kebetulan cocok (overfitting).

In [6]:
candidates_passing = grid_valid[grid_valid["profit_factor"] > 1.5]
print(f"Kandidat TRAIN PF>1.5 (sample>={MIN_SAMPLE_TRAIN}): {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat lolos kriteria TRAIN. Validasi TEST DIBATALKAN (kriteria kejujuran).")
else:
    test_results = []
    for _, row in candidates_passing.head(20).iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v24(df_test, **params, **FIXED)
        transition_test = trades_test[trades_test["mode"] == "TRANSITION"]
        m_test = evaluate(transition_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk semua kandidat TRAIN yg lolos (top 20) ===")
    print(test_df.to_string(index=False))

    robust = test_df[(test_df["test_pf"] > 1.5) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat yang ROBUST (lolos TRAIN >1.5 DAN TEST >1.5 DAN sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN PF>1.5 (sample>=30): 0

>>> TIDAK ADA kandidat lolos kriteria TRAIN. Validasi TEST DIBATALKAN (kriteria kejujuran).


## 7. Eksperimen BARU: RSI Ekstrem + Divergence (ganti BOS, yg terbukti gagal total)

**BOS (breakout struktur) terbukti gagal menyeluruh** -- 2304 kombinasi, semua PF<1.5.
Kesimpulan: logika "tangkap breakout awal via structure break" itu sendiri tidak py edge
yg cukup, terlepas parameter di sekitarnya. Ganti pendekatan sepenuhnya: alih-alih breakout,
coba **mean-reversion berbasis RSI ekstrem + divergence** -- beda logika fundamental (bukan
"ikut breakout", tapi "harga akan berbalik krn momentum oscillator sudah jenuh & TIDAK
dikonfirmasi harga").

**Kenapa RSI ekstrem SAJA blm cukup (pelajaran dari v20)**: v20 sudah coba BB %B/RSI ekstrem
sendirian utk mean-reversion di kondisi Ranging-Tenang -- GAGAL (PF terbaik 1.32, masih
banyak false-signal). **Divergence** menambah syarat lebih ketat: bukan cuma "RSI rendah",
tapi "harga bikin low BARU tapi RSI-nya TIDAK ikut turun" (`rsi_bull_div`) -- indikasi
momentum turun sudah melemah, potensi pembalikan lebih kuat drpd RSI rendah semata.

**Kandidat sinyal**: `rsi_bull_div=1` (harga Lower Low, RSI Higher Low) -> BUY; `rsi_bear_div=1`
(harga Higher High, RSI Lower High) -> SELL. Opsional dikombinasikan dgn syarat RSI juga di
area ekstrem (mis. RSI<40 saat bull_div, RSI>60 saat bear_div) -- grid search akan cari mana
yg lebih baik.

In [ ]:
RSI_DIV_CACHE_PATH = PROCESSED_DIR / "df_2019_2026_rsi_div.parquet"

if RSI_DIV_CACHE_PATH.exists():
    print(f"Load dari cache: {RSI_DIV_CACHE_PATH}")
    df_rsi = pd.read_parquet(RSI_DIV_CACHE_PATH)
else:
    print("Ambil kolom RSI/divergence dari CSV mentah, gabung ke df v23...")
    df_extra = pd.read_csv(
        PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv",
        usecols=["datetime", "rsi", "rsi_bull_div", "rsi_bear_div", "rsi_hid_bull", "rsi_hid_bear"],
    )
    df_extra["datetime"] = pd.to_datetime(df_extra["datetime"])
    df_rsi = df.merge(df_extra, on="datetime", how="left")
    df_rsi.to_parquet(RSI_DIV_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {RSI_DIV_CACHE_PATH}")

print(f"Total candle: {len(df_rsi)}")
print(f"rsi_bull_div=1: {(df_rsi['rsi_bull_div']==1).sum()} candle, "
      f"rsi_bear_div=1: {(df_rsi['rsi_bear_div']==1).sum()} candle")

In [ ]:
def run_backtest_rsi_div(
    df_signals: pd.DataFrame,
    adx_trend_min: float = 18.0,
    adx_transition_min: float = 0.0,
    adx_transition_max: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult_trend: float = 2.0,
    tp_mult_trend: float = 4.0,
    sl_mult_div: float = 1.5,
    tp_mult_div: float = 2.0,
    rsi_extreme_low: float = 100.0,   # syarat tambahan RSI juga ekstrem saat bull_div (100=tanpa syarat)
    rsi_extreme_high: float = 0.0,    # syarat tambahan RSI juga ekstrem saat bear_div (0=tanpa syarat)
    max_hold_trend: int = 12,
    max_hold_div: int = 12,
    enable_div: bool = True,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    rsi_arr = df_signals["rsi"].to_numpy()
    bull_div_arr = df_signals["rsi_bull_div"].to_numpy()
    bear_div_arr = df_signals["rsi_bear_div"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue

        direction = None
        mode = None
        sl_mult = tp_mult = max_hold = None

        if adx >= adx_trend_min:
            if score >= min_signal_score:
                direction = "BUY"
            elif score <= -min_signal_score:
                direction = "SELL"
            if direction is not None:
                if require_ob_filter:
                    opposing_ob = (
                        (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                        (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
                    )
                    if opposing_ob:
                        direction = None
                if direction is not None and require_h1_alignment:
                    if not check_h1_alignment_v24(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                        direction = None
            mode = "TREND"
            sl_mult, tp_mult, max_hold = sl_mult_trend, tp_mult_trend, max_hold_trend

        elif adx_transition_min <= adx < adx_transition_max and enable_div:
            rsi = rsi_arr[i]
            cand_dir = None
            if bull_div_arr[i] == 1 and np.isfinite(rsi) and rsi <= rsi_extreme_low:
                cand_dir = "BUY"
            elif bear_div_arr[i] == 1 and np.isfinite(rsi) and rsi >= rsi_extreme_high:
                cand_dir = "SELL"
            if cand_dir is not None:
                direction, mode = cand_dir, "RSI_DIV"
            sl_mult, tp_mult, max_hold = sl_mult_div, tp_mult_div, max_hold_div

        if direction is None:
            i += 1
            continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)

print("Backtest engine RSI divergence siap.")

In [ ]:
df_rsi_train = df_rsi[df_rsi["datetime"] < TRAIN_END].reset_index(drop=True)
df_rsi_test = df_rsi[df_rsi["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN: {len(df_rsi_train)} candle | TEST: {len(df_rsi_test)} candle")

import itertools
import time as _time

GRID_RSI = {
    "adx_transition_max": [15.0, 18.0, 22.0, 25.0],
    "sl_mult_div": [1.0, 1.5, 2.0, 3.0],
    "tp_mult_div": [1.0, 1.5, 2.0, 3.0, 4.0],
    "rsi_extreme_low": [100.0, 45.0, 40.0, 35.0, 30.0],   # 100 = tanpa syarat tambahan
    "max_hold_div": [6, 12, 20],
}
# rsi_extreme_high = kebalikan simetris dari rsi_extreme_low (100-x), dihitung inline

combos_rsi = list(itertools.product(*GRID_RSI.values()))
print(f"Total kombinasi grid RSI divergence: {len(combos_rsi)}")

t0 = _time.time()
grid_rsi_results = []
for idx, combo in enumerate(combos_rsi):
    params = dict(zip(GRID_RSI.keys(), combo))
    rsi_low = params.pop("rsi_extreme_low")
    rsi_high = 100.0 - rsi_low if rsi_low < 100.0 else 0.0
    trades = run_backtest_rsi_div(df_rsi_train, adx_transition_min=0.0, rsi_extreme_low=rsi_low, rsi_extreme_high=rsi_high, **params)
    div_only = trades[trades["mode"] == "RSI_DIV"]
    metrics = evaluate(div_only, INITIAL_EQUITY)
    metrics.update(params)
    metrics["rsi_extreme_low"] = rsi_low
    grid_rsi_results.append(metrics)
    if (idx + 1) % 100 == 0:
        print(f"  [{idx+1}/{len(combos_rsi)}] {_time.time()-t0:.0f}s")

grid_rsi_df = pd.DataFrame(grid_rsi_results)
print(f"\nGrid search RSI divergence selesai dalam {_time.time()-t0:.0f}s")

grid_rsi_valid = grid_rsi_df[grid_rsi_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 15 kandidat RSI_DIV (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_rsi_valid.head(15).to_string(index=False))
print(f"\nTotal kandidat sample cukup: {len(grid_rsi_valid)} dari {len(grid_rsi_df)}")
print(f"Kandidat dgn PF > 1.5: {(grid_rsi_valid['profit_factor'] > 1.5).sum()}")

In [ ]:
rsi_candidates_passing = grid_rsi_valid[grid_rsi_valid["profit_factor"] > 1.5]
print(f"Kandidat RSI_DIV TRAIN PF>1.5 (sample>={MIN_SAMPLE_TRAIN}): {len(rsi_candidates_passing)}")

if len(rsi_candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat lolos kriteria TRAIN. Validasi TEST DIBATALKAN.")
else:
    rsi_test_results = []
    for _, row in rsi_candidates_passing.head(20).iterrows():
        rsi_low = row["rsi_extreme_low"]
        rsi_high = 100.0 - rsi_low if rsi_low < 100.0 else 0.0
        params = {k: row[k] for k in ["adx_transition_max", "sl_mult_div", "tp_mult_div", "max_hold_div"]}
        trades_test = run_backtest_rsi_div(df_rsi_test, adx_transition_min=0.0, rsi_extreme_low=rsi_low, rsi_extreme_high=rsi_high, **params)
        div_test = trades_test[trades_test["mode"] == "RSI_DIV"]
        m_test = evaluate(div_test, INITIAL_EQUITY)
        rsi_test_results.append({**params, "rsi_extreme_low": rsi_low, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                                  "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                                  "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"]})

    rsi_test_df = pd.DataFrame(rsi_test_results)
    print("\n=== Validasi TEST kandidat RSI_DIV (top 20 dari TRAIN) ===")
    print(rsi_test_df.to_string(index=False))

    robust_rsi = rsi_test_df[(rsi_test_df["test_pf"] > 1.5) & (rsi_test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat RSI_DIV ROBUST (TRAIN & TEST PF>1.5, sample TEST>={MIN_SAMPLE_TEST}): {len(robust_rsi)}")
    if len(robust_rsi) > 0:
        print(robust_rsi.to_string(index=False))

## 6. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-5 -- placeholder)*